# Bicep Curl Analysis System - LIVE v6.2 (Webcam)



**Henry Abimanyu Dewandani | 1301220283 | Universitas Telkom**

> Jalankan cell berurutan. Jendela live: tekan Q untuk berhenti. Test di venue sebelum hari H.

In [1]:
# ============================================================
# CELL 1 -- Instalasi Dependensi (Lokal / VS Code)
# ============================================================

# Install semua dependensi yang dibutuhkan
!pip install opencv-python mediapipe==0.10.14 numpy pandas scipy protobuf==4.25.8 --timeout 300 -q
!pip install openpyxl

print("Instalasi selesai")

^C
Instalasi selesai



[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: C:\Users\ASUS\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: C:\Users\ASUS\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [1]:
# ============================================================
# CELL 2 - Imports
# ============================================================
import cv2
import mediapipe as mp
import numpy as np
import pandas as pd
import time
import os
import math
from collections import deque
from scipy.signal import savgol_filter
from IPython.display import HTML, display, clear_output
from base64 import b64encode

print(" Imports OK")

 Imports OK


In [2]:
# ============================================================
# CELL 3 - Helper Functions
# ============================================================

def calculate_angle(a, b, c):
    """
    Hitung sudut siku menggunakan atan2 (tiga titik: bahu-siku-pergelangan).
    Menghasilkan sudut dalam derajat [0, 180].
    Referensi: BAB III.3.5.2 Modul Perhitungan Sudut Siku
    """
    a, b, c = np.array(a), np.array(b), np.array(c)
    radians = np.arctan2(c[1]-b[1], c[0]-b[0]) - np.arctan2(a[1]-b[1], a[0]-b[0])
    angle = np.abs(radians * 180.0 / np.pi)
    if angle > 180.0:
        angle = 360.0 - angle
    return float(angle)


def smooth_angle_savgol(angle_buffer, window=7, polyorder=2):
    """
    Savitzky-Golay filter untuk smoothing sinyal sudut.
    Keunggulan vs moving average: mempertahankan puncak & lembah (peak/valley)
    yang krusial untuk deteksi ROM dan repetisi.
    Catatan: butuh minimal `window` data points; fallback ke mean jika kurang.
    """
    arr = np.array(angle_buffer)
    if len(arr) < window:
        return float(np.mean(arr))
    smoothed = savgol_filter(arr, window_length=window, polyorder=polyorder)
    return float(smoothed[-1])  # nilai terbaru


def draw_angle_arc(image, b_px, a_px, c_px, angle, color=(0, 255, 255), radius=40):
    """
    Gambar arc sudut di titik siku (b_px) menggunakan dua vektor:
    - vektor ke bahu (a_px)
    - vektor ke pergelangan (c_px)
    Menampilkan visualisasi sudut langsung di sendi.
    """
    try:
        v1 = np.array(a_px, dtype=float) - np.array(b_px, dtype=float)
        v2 = np.array(c_px, dtype=float) - np.array(b_px, dtype=float)
        angle1 = math.degrees(math.atan2(v1[1], v1[0]))
        angle2 = math.degrees(math.atan2(v2[1], v2[0]))
        start_a = min(angle1, angle2)
        end_a   = max(angle1, angle2)
        if end_a - start_a > 180:
            start_a, end_a = end_a, start_a + 360
        cv2.ellipse(image, tuple(map(int, b_px)),
                    (radius, radius), 0,
                    start_a, end_a, color, 2)
        # Label sudut di dekat siku
        label_pos = (int(b_px[0]) + radius + 5, int(b_px[1]))
        cv2.putText(image, f"{int(angle)}d", label_pos,
                    cv2.FONT_HERSHEY_SIMPLEX, 0.55, color, 2)
    except Exception:
        pass


def get_active_side(lm, mp_pose):
    """
    Auto-detect sisi aktif (kiri/kanan) berdasarkan rata-rata visibility
    dari 3 landmark utama per sisi.
    Mengembalikan: ('RIGHT'/'LEFT', shoulder_lm, elbow_lm, wrist_lm, hip_lm)
    """
    r_vis = (
        lm[mp_pose.PoseLandmark.RIGHT_SHOULDER.value].visibility +
        lm[mp_pose.PoseLandmark.RIGHT_ELBOW.value].visibility +
        lm[mp_pose.PoseLandmark.RIGHT_WRIST.value].visibility
    ) / 3

    l_vis = (
        lm[mp_pose.PoseLandmark.LEFT_SHOULDER.value].visibility +
        lm[mp_pose.PoseLandmark.LEFT_ELBOW.value].visibility +
        lm[mp_pose.PoseLandmark.LEFT_WRIST.value].visibility
    ) / 3

    if r_vis >= l_vis:
        side = 'RIGHT'
        s_lm = lm[mp_pose.PoseLandmark.RIGHT_SHOULDER.value]
        e_lm = lm[mp_pose.PoseLandmark.RIGHT_ELBOW.value]
        w_lm = lm[mp_pose.PoseLandmark.RIGHT_WRIST.value]
        h_lm = lm[mp_pose.PoseLandmark.RIGHT_HIP.value]
    else:
        side = 'LEFT'
        s_lm = lm[mp_pose.PoseLandmark.LEFT_SHOULDER.value]
        e_lm = lm[mp_pose.PoseLandmark.LEFT_ELBOW.value]
        w_lm = lm[mp_pose.PoseLandmark.LEFT_WRIST.value]
        h_lm = lm[mp_pose.PoseLandmark.LEFT_HIP.value]

    return side, s_lm, e_lm, w_lm, h_lm


def evaluate_feedback(smooth_angle, velocity, stability_std,
                      shoulder_swing, rom_current, shallow_flexion=False):
    """
    Rule-based feedback engine - 5 kondisi evaluasi teknik.
    Priority: kondisi buruk override 'Gerakan Bagus'.
    Referensi: BAB III.3.5.5 Modul Analisis Kualitas Gerakan

    Returns:
        feedback_text (str): pesan feedback
        color (tuple BGR): warna untuk overlay
        severity (str): 'good'/'warn'/'bad'
    """
    issues = []

    # Kondisi 1 - Terlalu cepat (momentum/cheating)
    if velocity > 180.0:
        issues.append(("Terlalu Cepat! Kurangi momentum", (0, 0, 255), 'bad'))

    # Kondisi 2 - Getaran/jitter berlebihan
    elif velocity < 20 and stability_std > 3.5:
        issues.append(("Stabilkan Lengan!", (0, 165, 255), 'warn'))

    # Kondisi 3 - Ayunan bahu (cheating/swing)
    if shoulder_swing > 15.0:
        issues.append(("Bahu Berayun! Jaga posisi", (0, 0, 255), 'bad'))

    # Kondisi 4 - ROM tidak penuh (di posisi puncak gerakan)
    if rom_current is not None and rom_current < 100:
        issues.append((f"ROM Kurang: {int(rom_current)}deg (ideal >120)", (0, 165, 255), 'warn'))

    # Kondisi 5 - Ekstensi tidak penuh (siku tidak lurus)
    # Batas bawah 120° mencegah false positive saat posisi fleksi penuh (angle ~30-50°)
    # Hanya aktif saat lengan di zona "hampir lurus tapi belum penuh" (120°-154°)
    if 120 < smooth_angle < 155 and velocity < 15:
        issues.append(("Luruskan Siku Penuh!", (0, 165, 255), 'warn'))

    # Kondisi 6 (v6.2) - Fleksi tidak penuh (percobaan rep gagal)
    if shallow_flexion:
        issues.insert(0, ("Fleksi Kurang Dalam! Angkat lebih tinggi", (0, 165, 255), 'warn'))

    if not issues:
        return "Gerakan Bagus!", (0, 220, 0), 'good'

    # Ambil issue dengan severity tertinggi
    bad_issues = [i for i in issues if i[2] == 'bad']
    if bad_issues:
        return bad_issues[0][0], bad_issues[0][1], 'bad'
    return issues[0][0], issues[0][1], 'warn'


print(' Helper functions loaded')

# ============================================================
# HELPER TAMBAHAN v6.0
# ============================================================

class AdaptiveThreshold:
    """
    Adaptive thresholding berdasarkan ROM aktual subjek.
    Menggunakan N rep pertama sebagai kalibrasi.
    Referensi: BAB III.3.5.4 Modul Deteksi Repetisi
    """
    def __init__(self, calibration_reps=2, buffer=10,
                 fallback_down=155, fallback_up=55):
        self.calibration_reps = calibration_reps
        self.buffer           = buffer
        self.threshold_down   = fallback_down  # ekstensi (sudut besar)
        self.threshold_up     = fallback_up    # fleksi (sudut kecil)
        self.calibrated       = False
        self._calib_angles    = []
        self._calib_done      = False

    def collect(self, angle, rep_count):
        if not self._calib_done and rep_count <= self.calibration_reps:
            self._calib_angles.append(angle)
        if not self._calib_done and rep_count > self.calibration_reps:
            if len(self._calib_angles) > 10:
                obs_min = min(self._calib_angles)
                obs_max = max(self._calib_angles)
                self.threshold_up   = obs_min + self.buffer   # fleksi threshold
                self.threshold_down = obs_max - self.buffer   # ekstensi threshold
                self.calibrated     = True
                print(f"\n[AdaptiveThreshold] Kalibrasi selesai ({self.calibration_reps} rep):")
                print(f"  UP threshold  (fleksi)   : {self.threshold_up:.1f} deg")
                print(f"  DOWN threshold (ekstensi): {self.threshold_down:.1f} deg")
            self._calib_done = True

    def get(self):
        return self.threshold_down, self.threshold_up

    def label(self):
        state = "adaptive" if self.calibrated else "default"
        return f"TH D:{self.threshold_down:.0f} U:{self.threshold_up:.0f} ({state})"


class ElbowLateralTracker:
    """
    Mengukur pergeseran horizontal siku relatif ke posisi awal setiap rep.
    Nilai dalam normalized coordinates [0,1] (x MediaPipe).
    Referensi: BAB III.3.5.5 Skenario 2 - pergeseran siku
    """
    def __init__(self, shift_threshold=0.04):
        self.shift_threshold = shift_threshold  # ~20-30px pada 720p
        self._baseline_x     = None
        self._prev_stage     = None

    def update(self, elbow_x, stage):
        # Ambil baseline saat transisi masuk ke stage 'turun' (ekstensi)
        if stage == "turun" and self._prev_stage != "turun":
            self._baseline_x = elbow_x
        self._prev_stage = stage

        if self._baseline_x is None:
            return 0.0, False
        shift      = abs(elbow_x - self._baseline_x)
        is_shifted = shift > self.shift_threshold
        return round(shift, 4), is_shifted


class TUTTracker:
    """
    Menghitung Time Under Tension (detik) per repetisi.
    Konsentrik: stage turun -> naik (angkat beban)
    Eksentrik : stage naik -> turun (turunkan beban)
    Referensi: BAB III.3.5.5 Modul Analisis Kualitas Gerakan
    """
    def __init__(self):
        self._prev_stage          = None
        self._concentric_start_ms = None
        self._eccentric_start_ms  = None
        self.tut_concentric_ms    = 0.0
        self.tut_eccentric_ms     = 0.0
        self._current_rep         = 0
        self.rep_tut_log          = []

    def update(self, stage, timestamp_ms, rep_count):
        if self._prev_stage != stage:
            if self._prev_stage == "turun" and stage == "naik":
                # Mulai fase konsentrik
                self._concentric_start_ms = timestamp_ms
                if self._eccentric_start_ms is not None:
                    self.tut_eccentric_ms = timestamp_ms - self._eccentric_start_ms
                    self._eccentric_start_ms = None

            elif self._prev_stage == "naik" and stage == "turun":
                # Mulai fase eksentrik
                self._eccentric_start_ms = timestamp_ms
                if self._concentric_start_ms is not None:
                    self.tut_concentric_ms = timestamp_ms - self._concentric_start_ms
                    self._concentric_start_ms = None

                # Rep selesai saat kembali ke turun
                if rep_count > self._current_rep:
                    self._current_rep = rep_count
                    total_s = (self.tut_concentric_ms + self.tut_eccentric_ms) / 1000.0
                    self.rep_tut_log.append({
                        'rep'             : rep_count,
                        'tut_concentric_s': round(self.tut_concentric_ms / 1000.0, 2),
                        'tut_eccentric_s' : round(self.tut_eccentric_ms  / 1000.0, 2),
                        'tut_total_s'     : round(total_s, 2),
                    })
        self._prev_stage = stage

    def elapsed(self, timestamp_ms):
        if self._concentric_start_ms is not None:
            return round((timestamp_ms - self._concentric_start_ms) / 1000.0, 2)
        if self._eccentric_start_ms is not None:
            return round((timestamp_ms - self._eccentric_start_ms) / 1000.0, 2)
        return 0.0

    def print_summary(self):
        import numpy as np
        print("\n" + "="*52)
        print("   TIME UNDER TENSION (TUT) SUMMARY")
        print("="*52)
        if not self.rep_tut_log:
            print("  Tidak ada data TUT (kurang dari 1 rep penuh)")
            return
        print(f"{'Rep':>4}  {'Konsentrik':>12}  {'Eksentrik':>11}  {'Total':>7}")
        print("-"*44)
        for r in self.rep_tut_log:
            print(f"{r['rep']:>4}  {r['tut_concentric_s']:>10.2f}s  "
                  f"{r['tut_eccentric_s']:>9.2f}s  {r['tut_total_s']:>5.2f}s")
        print("-"*44)
        totals = [r['tut_total_s'] for r in self.rep_tut_log]
        avg    = float(np.mean(totals))
        print(f"  Total TUT  : {sum(totals):.2f}s")
        print(f"  Avg per rep: {avg:.2f}s")
        if avg < 2.0:
            print("  PERINGATAN: Rata-rata TUT < 2s - gerakan terlalu cepat (momentum).")
        elif avg > 6.0:
            print("  INFO: TUT > 6s/rep - tempo sangat lambat/terkontrol.")
        else:
            print("  INFO: Tempo normal (2–6s per rep).")


print(' Helper functions v6.0 loaded (AdaptiveThreshold, ElbowLateralTracker, TUTTracker)')


 Helper functions loaded
 Helper functions v6.0 loaded (AdaptiveThreshold, ElbowLateralTracker, TUTTracker)


In [3]:
# ============================================================
# CELL 4 - Configuration & Pilih Video
# ============================================================

# --- Parameter Thresholding (Adaptive v6.0) ---
# Nilai berikut adalah default fallback selama kalibrasi (2 rep pertama)
# Setelah rep ke-2, threshold otomatis menyesuaikan dengan ROM subjek
DOWN_THRESHOLD   = 155   # fallback ekstensi
UP_THRESHOLD     = 55    # fallback fleksi
CALIBRATION_REPS = 2     # jumlah rep untuk kalibrasi adaptive threshold
TH_BUFFER        = 10    # buffer margin dari observed min/max (derajat)
ELBOW_SHIFT_TH   = 0.04  # normalized threshold pergeseran siku (~20-30px pada 720p)
SPEED_LIMIT      = 180.0 # deg/s - batas kecepatan sebelum dianggap terlalu cepat
STABILITY_LIMIT  = 3.5   # std dev derajat - batas jitter
SWING_LIMIT      = 15.0  # derajat perubahan bahu - batas cheating
MIN_VISIBILITY   = 0.6   # skor minimum agar frame dianggap valid
SG_WINDOW        = 9     # Savitzky-Golay window (harus ganjil, >= polyorder+2)
SG_POLY          = 2     # Savitzky-Golay polynomial order

# --- Output folder ---
OUTPUT_DIR = "./outputs_live"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# --- Inisialisasi webcam ---
CAMERA_INDEX = 0   # ubah ke 1/2 jika pakai webcam eksternal
cap_test = cv2.VideoCapture(CAMERA_INDEX)
if not cap_test.isOpened():
    raise RuntimeError(f"Webcam index {CAMERA_INDEX} tidak dapat dibuka!")
cap_test.release()
print(f" Webcam index {CAMERA_INDEX} siap. Tekan Q untuk berhenti.")

 Webcam index 0 siap. Tekan Q untuk berhenti.


In [4]:
# ============================================================
# CELL 5 - Main Processing Pipeline
# ============================================================

# --- Buka video & siapkan writer ---
cap = cv2.VideoCapture(CAMERA_INDEX)
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 1280)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)
fps = cap.get(cv2.CAP_PROP_FPS)
if fps == 0 or np.isnan(fps):
    fps = 30.0
width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))  or 640
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT)) or 480

out = None

total_frames = 99999  # live mode: tidak diketahui

out_video_path = None  # live mode: tidak simpan video

# --- MediaPipe setup ---
mp_pose    = mp.solutions.pose
mp_drawing = mp.solutions.drawing_utils
mp_drawing_styles = mp.solutions.drawing_styles

# --- State variables ---
angle_buffer      = deque(maxlen=SG_WINDOW + 2)
shoulder_y_buffer = deque(maxlen=10)
data_rows         = []
rep_data          = []

rep_count         = 0
stage             = "netral"
prev_time         = 0.0
prev_smooth_angle = 0.0
active_side       = "?"

rep_min_angle         = 999.0
rep_max_angle         = 0.0
rep_angles_inprogress = []
current_rom           = None
last_shoulder_y       = None
shoulder_swing        = 0.0

# --- v6.0: Inisialisasi modul tambahan ---
adaptive_th   = AdaptiveThreshold(CALIBRATION_REPS, TH_BUFFER, DOWN_THRESHOLD, UP_THRESHOLD)
elbow_tracker = ElbowLateralTracker(ELBOW_SHIFT_TH)
tut_tracker   = TUTTracker()
elbow_shift   = 0.0
elbow_shifted = False
tut_elapsed   = 0.0

print(f" Video info: {width}x{height} @ {fps:.1f}fps | {total_frames} frames ({total_frames/fps:.1f}s)")
print(f"  Config: DOWN>{DOWN_THRESHOLD}° | UP<{UP_THRESHOLD}° | MaxSpeed={SPEED_LIMIT}°/s")
print(" Processing...\n")

# --- v6.2: deteksi percobaan rep tidak sah ---
FAILED_DIP_MIN   = 40.0
SHALLOW_RISE_MIN = 15.0
REBOUND_MARGIN   = 30.0
attempt_min_angle   = 999.0
prev_in_top         = False
prev_in_bottom      = False
bottom_armed        = True
failed_attempts     = []
shallow_flexion_now = False

with mp_pose.Pose(min_detection_confidence=0.6, min_tracking_confidence=0.6) as pose:
    frame_idx = 0

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        frame = cv2.flip(frame, 1)

        frame_idx += 1
        timestamp_ms  = cap.get(cv2.CAP_PROP_POS_MSEC)
        current_time  = timestamp_ms / 1000.0

        # Live mode: print status setiap 60 frame
        if frame_idx % 60 == 0:
            print(f"\r  Frame {frame_idx} | Reps: {rep_count} | Gagal: {len(failed_attempts)} | Angle: {int(smooth_angle)}°", end='')

        # --- Pose detection ---
        image_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results   = pose.process(image_rgb)
        image     = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2BGR)

        # Default values (frame tanpa pose)
        raw_angle      = 0.0
        smooth_angle   = prev_smooth_angle
        velocity       = 0.0
        stability_std  = 0.0
        avg_visibility = 0.0
        feedback_text  = "Mencari Pose..."
        fb_color       = (200, 200, 200)
        pose_valid     = False

        if results.pose_landmarks:
            lm = results.pose_landmarks.landmark

            # --- Auto side detection ---
            side, s_lm, e_lm, w_lm, h_lm = get_active_side(lm, mp_pose)
            active_side = side

            avg_visibility = (s_lm.visibility + e_lm.visibility + w_lm.visibility) / 3

            if avg_visibility >= MIN_VISIBILITY:
                pose_valid = True

                # v6.2: koreksi rasio aspek - koordinat dikonversi ke pixel
                shoulder = [s_lm.x * width, s_lm.y * height]
                elbow    = [e_lm.x * width, e_lm.y * height]
                wrist    = [w_lm.x * width, w_lm.y * height]

                # --- Kalkulasi sudut ---
                raw_angle = calculate_angle(shoulder, elbow, wrist)
                angle_buffer.append(raw_angle)
                smooth_angle = smooth_angle_savgol(angle_buffer, SG_WINDOW, SG_POLY)

                # --- Savitzky-Golay vs v4.1 moving average ---
                # SG filter mempertahankan bentuk kurva asli lebih baik
                # sehingga peak/valley untuk ROM dan repetisi lebih akurat

                # --- Metrik ---
                stability_std = float(np.std(angle_buffer)) if len(angle_buffer) > 1 else 0.0
                dt = current_time - prev_time
                velocity = abs(smooth_angle - prev_smooth_angle) / dt if dt > 0 else 0.0

                # --- Deteksi shoulder swing (cheating) ---
                shoulder_y_buffer.append(s_lm.y)
                if len(shoulder_y_buffer) > 3:
                    shoulder_swing = float(np.std(shoulder_y_buffer)) * 1000  # normalisasi ke ~derajat
                    # Catatan: koordinat MediaPipe adalah normalized [0,1],
                    # perkalian 1000 mengkonversi ke satuan yang sebanding
                else:
                    shoulder_swing = 0.0

                # --- v6.0: Update adaptive threshold & modul tambahan ---
                adaptive_th.collect(smooth_angle, rep_count)
                DOWN_THRESHOLD_ACT, UP_THRESHOLD_ACT = adaptive_th.get()

                # --- Elbow lateral shift ---
                elbow_shift, elbow_shifted = elbow_tracker.update(e_lm.x, stage)

                # --- Tracking ROM dalam progress rep ---
                rep_angles_inprogress.append(smooth_angle)
                rep_min_angle = min(rep_min_angle, smooth_angle)
                rep_max_angle = max(rep_max_angle, smooth_angle)

                # --- Deteksi fase & repetisi (adaptive threshold) ---
                # Referensi: BAB III.3.5.3 & 3.5.4
                rep_just_counted = False
                if smooth_angle > DOWN_THRESHOLD_ACT:
                    stage = "turun"   # posisi ekstensi

                if smooth_angle < UP_THRESHOLD_ACT and stage == "turun":
                    stage = "naik"    # posisi fleksi puncak
                    rep_count += 1
                    rep_just_counted = True

                    # Hitung & simpan ROM untuk rep ini
                    current_rom = rep_max_angle - rep_min_angle
                    rep_data.append({
                        "rep": rep_count,
                        "rom_deg": round(current_rom, 1),
                        "min_angle": round(rep_min_angle, 1),
                        "max_angle": round(rep_max_angle, 1),
                        "avg_velocity": round(
                            float(np.mean([abs(rep_angles_inprogress[i]-rep_angles_inprogress[i-1])
                                          for i in range(1, len(rep_angles_inprogress))])
                                  * fps), 1
                        ) if len(rep_angles_inprogress) > 1 else 0.0,
                        "timestamp_s": round(current_time, 2)
                    })

                    # Reset tracking untuk rep berikutnya
                    rep_min_angle = 999.0
                    rep_max_angle = 0.0
                    rep_angles_inprogress = []

                # --- v6.0: TUT update (setiap frame) ---
                # --- v6.2: deteksi percobaan rep tidak sah ---
                in_top    = smooth_angle > DOWN_THRESHOLD_ACT
                in_bottom = smooth_angle < UP_THRESHOLD_ACT
                if prev_in_top and not in_top:
                    attempt_min_angle = smooth_angle
                if not in_top:
                    attempt_min_angle = min(attempt_min_angle, smooth_angle)
                if in_top and not prev_in_top:
                    if UP_THRESHOLD_ACT < attempt_min_angle <= (DOWN_THRESHOLD_ACT - FAILED_DIP_MIN):
                        failed_attempts.append({
                            "attempt": len(failed_attempts) + 1,
                            "alasan": "fleksi tidak penuh",
                            "min_angle": round(attempt_min_angle, 1),
                            "timestamp_s": round(current_time, 2)})
                    attempt_min_angle = 999.0
                if smooth_angle > UP_THRESHOLD_ACT + REBOUND_MARGIN:
                    bottom_armed = True
                if (in_bottom and not prev_in_bottom and stage == "naik"
                        and not rep_just_counted and bottom_armed):
                    failed_attempts.append({
                        "attempt": len(failed_attempts) + 1,
                        "alasan": "ekstensi tidak penuh",
                        "min_angle": round(smooth_angle, 1),
                        "timestamp_s": round(current_time, 2)})
                if in_bottom:
                    bottom_armed = False
                prev_in_top, prev_in_bottom = in_top, in_bottom
                shallow_flexion_now = (
                    stage == "turun" and not in_top
                    and attempt_min_angle < 900
                    and attempt_min_angle > UP_THRESHOLD_ACT
                    and attempt_min_angle <= (DOWN_THRESHOLD_ACT - FAILED_DIP_MIN)
                    and smooth_angle > attempt_min_angle + SHALLOW_RISE_MIN
                    and smooth_angle > prev_smooth_angle)

                tut_tracker.update(stage, timestamp_ms, rep_count)
                tut_elapsed = tut_tracker.elapsed(timestamp_ms)

                # --- Feedback engine ---
                feedback_text, fb_color, _ = evaluate_feedback(
                    smooth_angle, velocity, stability_std,
                    shoulder_swing, current_rom,
                    shallow_flexion=shallow_flexion_now
                )

                prev_time         = current_time
                prev_smooth_angle = smooth_angle

                # --- Draw landmarks ---
                mp_drawing.draw_landmarks(
                    image, results.pose_landmarks, mp_pose.POSE_CONNECTIONS,
                    mp_drawing.DrawingSpec(color=(245,117,66), thickness=2, circle_radius=3),
                    mp_drawing.DrawingSpec(color=(245,66,230), thickness=2)
                )

                # --- Arc sudut di siku ---
                s_px = (int(s_lm.x * width), int(s_lm.y * height))
                e_px = (int(e_lm.x * width), int(e_lm.y * height))
                w_px = (int(w_lm.x * width), int(w_lm.y * height))
                draw_angle_arc(image, e_px, s_px, w_px, smooth_angle)

        # --------------------------------------------------------
        # VISUALISASI OVERLAY - Panel info kiri atas
        # --------------------------------------------------------
        if image.shape[1] != 1280 or image.shape[0] != 720:
            image = cv2.resize(image, (1280, 720))

        panel_h = 390
        overlay = image.copy()
        cv2.rectangle(overlay, (5, 5), (445, panel_h), (0, 0, 0), -1)
        cv2.addWeighted(overlay, 0.6, image, 0.4, 0, image)  # transparan
        cv2.rectangle(image, (5, 5), (440, panel_h), (80, 80, 80), 1)

        font  = cv2.FONT_HERSHEY_SIMPLEX
        font2 = cv2.FONT_HERSHEY_DUPLEX

        # Header: side indicator
        side_color = (100, 200, 255) if active_side == 'RIGHT' else (255, 200, 100)
        shown_side = 'LEFT' if active_side == 'RIGHT' else 'RIGHT'
        cv2.putText(image, f"[{shown_side} ARM]", (1040, 40), font, 0.6, side_color, 2)

        # Baris 1 - Reps & Stage
        cv2.putText(image, f"Reps: {rep_count} | Gagal: {len(failed_attempts)} | {stage.upper()}",
                    (15, 40), font2, 0.85, (255, 255, 255), 2)

        # Baris 2 - Angle (raw & smooth)
        cv2.putText(image, f"Sudut Siku : {int(smooth_angle):>3}  (raw:{int(raw_angle):>3}) deg",
                    (15, 75), font, 0.65, (0, 255, 255), 2)

        # Baris 3 - Speed
        v_color = (0, 0, 255) if velocity > SPEED_LIMIT else (255, 255, 255)
        cv2.putText(image, f"Kecepatan  : {int(velocity):>4} deg/s",
                    (15, 108), font, 0.65, v_color, 2)

        # Baris 4 - Jitter
        s_color = (0, 0, 255) if stability_std > STABILITY_LIMIT else (255, 255, 255)
        cv2.putText(image, f"Stabilitas : {stability_std:>5.2f} (jitter)",
                    (15, 141), font, 0.65, s_color, 2)

        # Baris 5 - Shoulder swing
        sw_color = (0, 165, 255) if shoulder_swing > SWING_LIMIT else (255, 255, 255)
        cv2.putText(image, f"Bahu Swing : {shoulder_swing:>5.1f} (threshold:{SWING_LIMIT})",
                    (15, 174), font, 0.65, sw_color, 2)

        # Baris 6 - Elbow shift (v6.0)
        es_color = (0, 0, 255) if elbow_shifted else (255, 255, 255)
        cv2.putText(image, f"Siku Shift : {elbow_shift*100:>4.1f}% {'CHEATING!' if elbow_shifted else ''}",
                    (15, 207), font, 0.60, es_color, 2)

        # Baris 7 - TUT elapsed & adaptive threshold status (v6.0)
        cv2.putText(image, f"TUT Elapsed: {tut_elapsed:.1f}s  | {adaptive_th.label()}",
                    (15, 238), font, 0.50, (180, 200, 180), 1)

        # Baris 8 - ROM terakhir
        rom_str = f"{current_rom:.0f} deg" if current_rom is not None else "--"
        rom_color = (0, 165, 255) if (current_rom is not None and current_rom < 100) else (255, 255, 255)
        cv2.putText(image, f"ROM Terakhir: {rom_str}",
                    (15, 271), font, 0.65, rom_color, 2)

        # Baris 9 - Visibility
        cv2.putText(image, f"Visibility : {avg_visibility:.2f}",
                    (15, 304), font, 0.65, (180, 180, 180), 1)

        # Baris 10 - Feedback (paling bawah, highlight)
        cv2.rectangle(image, (5, 316), (440, panel_h), (30, 30, 30), -1)
        cv2.putText(image, feedback_text,
                    (15, 353), font, 0.75, fb_color, 2)

        # Fullscreen display
        WINDOW_NAME = "Bicep Curl Analysis v6 LIVE"
        if frame_idx == 1:
            cv2.namedWindow(WINDOW_NAME, cv2.WINDOW_NORMAL)
            cv2.setWindowProperty(WINDOW_NAME, cv2.WND_PROP_FULLSCREEN, cv2.WINDOW_FULLSCREEN)

        if out is None:
            h, w = image.shape[:2]
            out = cv2.VideoWriter('live_demo_output.avi',
                                  cv2.VideoWriter_fourcc(*'XVID'), 15.0, (w, h))
            print("Rekaman aktif:", out.isOpened(), "ukuran", w, "x", h)
        out.write(image)    
        cv2.imshow(WINDOW_NAME, image)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            print("\n Analisis dihentikan.")
            break

        # --- Simpan data frame ---
        data_rows.append({
            "frame"           : frame_idx,
            "timestamp_ms"   : round(timestamp_ms, 2),
            "time_s"         : round(current_time, 2),
            "active_side"    : active_side,
            "angle_raw"      : round(raw_angle, 2),
            "angle_smooth_sg": round(smooth_angle, 2),   # SG-filtered
            "velocity_degs"  : round(velocity, 2),
            "stability_jitter": round(stability_std, 2),
            "shoulder_swing" : round(shoulder_swing, 2),
            "visibility"     : round(avg_visibility, 4),
            "stage"          : stage,
            "rep_count"      : rep_count,
            "pose_valid"     : pose_valid,
            "feedback"           : feedback_text,
            # --- v6.0 additions ---
            "elbow_lateral_shift": elbow_shift,
            "elbow_cheating"     : int(elbow_shifted),
            "tut_elapsed_s"      : tut_elapsed,
            "threshold_down_act" : round(adaptive_th.get()[0], 1),
            "threshold_up_act"   : round(adaptive_th.get()[1], 1),
        })

cap.release()
if out is not None:
    out.release()
cv2.destroyAllWindows()
print(f"\n\n Processing selesai! Total rep terdeteksi: {rep_count}")
tut_tracker.print_summary()

 Video info: 1280x720 @ 30.0fps | 99999 frames (3333.3s)
  Config: DOWN>155° | UP<55° | MaxSpeed=180.0°/s
 Processing...



C:\Users\ASUS\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '


Rekaman aktif: True ukuran 1280 x 720
  Frame 1200 | Reps: 2 | Gagal: 5 | Angle: 152°
[AdaptiveThreshold] Kalibrasi selesai (2 rep):
  UP threshold  (fleksi)   : 48.3 deg
  DOWN threshold (ekstensi): 188.7 deg
  Frame 2340 | Reps: 5 | Gagal: 5 | Angle: 160°
 Analisis dihentikan.


 Processing selesai! Total rep terdeteksi: 5

   TIME UNDER TENSION (TUT) SUMMARY
 Rep    Konsentrik    Eksentrik    Total
--------------------------------------------
   1        0.87s       0.00s   0.87s
   2        1.47s       7.17s   8.63s
   3        0.43s       8.93s   9.37s
   4        2.13s       0.47s   2.60s
--------------------------------------------
  Total TUT  : 21.47s
  Avg per rep: 5.37s
  INFO: Tempo normal (2–6s per rep).


In [5]:
# ── Helper: export DataFrame ke Excel tabel terformat ─────────────────────────
def save_xlsx_table(df, path, sheet_name='Data', col_rename=None,
                    table_name='PoseData', style='TableStyleMedium9'):
    """Simpan DataFrame sebagai Excel dengan format tabel (banded rows, header bold)."""
    from openpyxl.utils import get_column_letter
    from openpyxl.worksheet.table import Table, TableStyleInfo

    df_out = df.copy()
    if col_rename:
        df_out = df_out.rename(columns=col_rename)

    with pd.ExcelWriter(path, engine='openpyxl') as writer:
        df_out.to_excel(writer, index=False, sheet_name=sheet_name)
        ws = writer.sheets[sheet_name]

        # Tambah tabel Excel
        n_rows   = len(df_out) + 1
        last_col = get_column_letter(len(df_out.columns))
        tab = Table(displayName=table_name, ref=f"A1:{last_col}{n_rows}")
        tab.tableStyleInfo = TableStyleInfo(
            name=style, showFirstColumn=False, showLastColumn=False,
            showRowStripes=True, showColumnStripes=False
        )
        ws.add_table(tab)

        # Auto-fit lebar kolom
        for col in ws.columns:
            col_letter = get_column_letter(col[0].column)
            max_w = max((len(str(c.value or '')) for c in col), default=10)
            ws.column_dimensions[col_letter].width = min(max_w + 4, 35)

    return path

FRAMES_COL_RENAME = {
    'frame'             : 'Frame',
    'timestamp_ms'      : 'Timestamp (ms)',
    'time_s'            : 'Waktu (s)',
    'active_side'       : 'Sisi Aktif',
    'angle_raw'         : 'Sudut Raw (°)',
    'angle_smooth_sg'   : 'Sudut Smooth SG (°)',
    'velocity_degs'     : 'Kecepatan (°/s)',
    'stability_jitter'  : 'Jitter (Stabilitas)',
    'shoulder_swing'    : 'Shoulder Swing',
    'visibility'        : 'Visibility',
    'stage'             : 'Stage',
    'rep_count'         : 'Rep Count',
    'pose_valid'        : 'Pose Valid',
    'feedback'          : 'Feedback',
    'elbow_lateral_shift': 'Elbow Shift',
    'elbow_cheating'    : 'Elbow Cheating',
    'tut_elapsed_s'     : 'TUT Elapsed (s)',
    'threshold_down_act': 'Threshold Down (°)',
    'threshold_up_act'  : 'Threshold Up (°)',
}
REPREP_COL_RENAME = {
    'rep'         : 'Rep',
    'rom_deg'     : 'ROM (°)',
    'min_angle'   : 'Sudut Min (°)',
    'max_angle'   : 'Sudut Maks (°)',
    'avg_velocity': 'Kec. Rata-rata (°/s)',
    'timestamp_s' : 'Waktu (s)',
}
TUT_COL_RENAME = {
    'rep'          : 'Rep',
    'concentric_s' : 'Konsentrik (s)',
    'eccentric_s'  : 'Eksentrik (s)',
    'total_tut_s'  : 'Total TUT (s)',
}
# ============================================================
# CELL 6 - Save CSV & XLSX (Terformat) + Ringkasan Sesi
# ============================================================

# --- CSV (backup) & XLSX (utama) frame-by-frame ---
df = pd.DataFrame(data_rows)
csv_path  = os.path.join(OUTPUT_DIR, "bicep_analysis_v6_frames.csv")
xlsx_path = os.path.join(OUTPUT_DIR, "bicep_analysis_v6_frames.xlsx")
df.to_csv(csv_path, index=False, encoding='utf-8-sig')
save_xlsx_table(df, xlsx_path, sheet_name='Frames', col_rename=FRAMES_COL_RENAME,
                table_name='FrameData')
print(f" Frame data: {csv_path}")
print(f"             {xlsx_path}")

# --- CSV (backup) & XLSX (utama) per-repetisi ---
df_rep = pd.DataFrame(rep_data)
csv_rep_path  = os.path.join(OUTPUT_DIR, "bicep_analysis_v6_per_rep.csv")
xlsx_rep_path = os.path.join(OUTPUT_DIR, "bicep_analysis_v6_per_rep.xlsx")
df_rep.to_csv(csv_rep_path, index=False, encoding='utf-8-sig')
save_xlsx_table(df_rep, xlsx_rep_path, sheet_name='Per_Rep', col_rename=REPREP_COL_RENAME,
                table_name='RepData', style='TableStyleMedium2')
print(f" Per-rep   : {csv_rep_path}")
print(f"             {xlsx_rep_path}")

# --- Ringkasan sesi ---
print("\n" + "="*55)
print("   RINGKASAN SESI LATIHAN - BICEP CURL")
print("="*55)
print(f"  Sisi aktif        : {active_side} ARM")
print(f"  Total repetisi    : {rep_count} reps")

if rep_data:
    roms = [r['rom_deg'] for r in rep_data]
    vels = [r['avg_velocity'] for r in rep_data]
    print(f"  ROM rata-rata     : {np.mean(roms):.1f}° (min:{min(roms):.1f}° max:{max(roms):.1f}°)")
    print(f"  ROM ideal (>120°) : {' Tercapai' if np.mean(roms) >= 120 else '⚠️  Perlu ditingkatkan'}")
    print(f"  Kecepatan avg     : {np.mean(vels):.1f} deg/s")
    print()
    print("  Detail per repetisi:")
    print(f"  {'Rep':>4} | {'ROM':>6} | {'Min°':>5} | {'Max°':>5} | {'AvgVel':>8} | {'Waktu':>6}")
    print(f"  {'-'*4}-+-{'-'*6}-+-{'-'*5}-+-{'-'*5}-+-{'-'*8}-+-{'-'*6}")
    for r in rep_data:
        rom_ok = '✅' if r['rom_deg'] >= 120 else '⚠️'
        print(f"  {r['rep']:>4} | {r['rom_deg']:>5.1f}° | {r['min_angle']:>4.1f}° | "
              f"{r['max_angle']:>4.1f}° | {r['avg_velocity']:>7.1f}°/s | {r['timestamp_s']:>5.1f}s  {rom_ok}")

# Feedback summary dari CSV
if len(df) > 0:
    fb_counts = df[df['pose_valid'] == True]['feedback'].value_counts()
    total_valid = df['pose_valid'].sum()
    print(f"\n  Distribusi feedback ({total_valid} frame valid):")
    for fb, cnt in fb_counts.items():
        pct = cnt / total_valid * 100
        print(f"    {fb:<35} : {cnt:>4} frames ({pct:.1f}%)")

print("="*55)
# --- CSV TUT per-repetisi (v6.0) ---
if tut_tracker.rep_tut_log:
    df_tut = pd.DataFrame(tut_tracker.rep_tut_log)
    csv_tut_path = os.path.join(OUTPUT_DIR, "bicep_analysis_v6_tut.csv")
    df_tut.to_csv(csv_tut_path, index=False, encoding='utf-8-sig')
    xlsx_tut_path = os.path.join(OUTPUT_DIR, "bicep_analysis_v6_tut.xlsx")
    save_xlsx_table(df_tut, xlsx_tut_path, sheet_name='TUT', col_rename=TUT_COL_RENAME,
                    table_name='TUTData', style='TableStyleLight9')
    print(f" TUT data  : {csv_tut_path}")
    print(f"             {xlsx_tut_path}")

# --- Fatigue Index (v6.0) ---
def compute_fatigue_index(df_frames):
    import numpy as np
    per_rep = df_frames[df_frames['rep_count'] > 0].groupby('rep_count').agg(
        angle_min    = ('angle_smooth_sg', 'min'),
        angle_max    = ('angle_smooth_sg', 'max'),
        avg_velocity = ('velocity_degs',   'mean'),
        avg_jitter   = ('stability_jitter','mean'),
    ).reset_index()
    per_rep['rom'] = per_rep['angle_max'] - per_rep['angle_min']
    if len(per_rep) < 3:
        print("[FatigueIndex] Butuh minimal 3 rep untuk menghitung tren.")
        return
    x = per_rep['rep_count'].values
    def slope_norm(y):
        c = np.polyfit(x, y, 1)
        return c[0] / (np.mean(y) + 1e-9)
    s_rom = slope_norm(per_rep['rom'].values)
    s_vel = slope_norm(per_rep['avg_velocity'].values)
    s_jit = slope_norm(per_rep['avg_jitter'].values)
    fi = min(1.0, (max(0, -s_rom) + max(0, -s_vel) + max(0, s_jit)) / 3.0 * 10)
    print("\n" + "="*52)
    print("   FATIGUE INDEX ANALYSIS")
    print("="*52)
    print(per_rep[['rep_count','rom','avg_velocity','avg_jitter']].to_string(index=False))
    print(f"\n  Slope ROM      : {s_rom:.4f}")
    print(f"  Slope Velocity : {s_vel:.4f}")
    print(f"  Slope Jitter   : {s_jit:.4f}")
    print(f"  Fatigue Index  : {fi:.3f}  (0=fresh, 1=sangat fatigue)")
    if fi > 0.6:
        print("  >> Tanda kelelahan signifikan terdeteksi.")
    elif fi > 0.3:
        print("  >> Kelelahan ringan - waspadai teknik di rep akhir.")
    else:
        print("  >> Tidak ada tanda kelelahan signifikan.")

compute_fatigue_index(df)


# --- v6.2: export percobaan tidak sah (jika ada) ---
if failed_attempts:
    df_fail = pd.DataFrame(failed_attempts)
    fail_path = os.path.join(OUTPUT_DIR, "bicep_live_failed_attempts.csv")
    df_fail.to_csv(fail_path, index=False, encoding="utf-8-sig")
    print(f"Percobaan tidak sah: {len(failed_attempts)} (tersimpan: {fail_path})")
else:
    print("Tidak ada percobaan tidak sah pada sesi ini.")

 Frame data: ./outputs_live\bicep_analysis_v6_frames.csv
             ./outputs_live\bicep_analysis_v6_frames.xlsx
 Per-rep   : ./outputs_live\bicep_analysis_v6_per_rep.csv
             ./outputs_live\bicep_analysis_v6_per_rep.xlsx

   RINGKASAN SESI LATIHAN - BICEP CURL
  Sisi aktif        : LEFT ARM
  Total repetisi    : 5 reps
  ROM rata-rata     : 154.1° (min:116.4° max:205.1°)
  ROM ideal (>120°) :  Tercapai
  Kecepatan avg     : 190.8 deg/s

  Detail per repetisi:
   Rep |    ROM |  Min° |  Max° |   AvgVel |  Waktu
  -----+--------+-------+-------+----------+-------
     1 | 146.8° | 51.8° | 198.7° |    96.4°/s | 16332.8s  ✅
     2 | 116.4° | 49.0° | 165.4° |    76.2°/s | 16340.8s  ⚠️
     3 | 137.3° | 38.3° | 175.6° |    38.8°/s | 16351.2s  ✅
     4 | 164.8° | 27.0° | 191.8° |   662.4°/s | 16352.1s  ✅
     5 | 205.1° | -13.7° | 191.4° |    80.3°/s | 16372.3s  ✅

  Distribusi feedback (1843 frame valid):
    Gerakan Bagus!                      : 1162 frames (63.0%)
    Luruskan S

In [6]:
# Export CSV setelah sesi selesai
import pandas as pd, numpy as np

print("\n" + "="*50)
print("   RINGKASAN SESI LIVE")
print("="*50)
if rep_data:
    roms = [r["rom_deg"] for r in rep_data]
    print(f"  Rep Terdeteksi  : {rep_count}")
    print(f"  ROM Rata-rata   : {np.mean(roms):.1f}°  (Min:{min(roms):.0f}° / Maks:{max(roms):.0f}°)")
else:
    print("  Tidak ada rep terdeteksi.")

if data_rows:
    import os
    pd.DataFrame(data_rows).to_csv(os.path.join(OUTPUT_DIR,"live_frames.csv"), index=False)
    print(f"  CSV disimpan di: {OUTPUT_DIR}/")
print("="*50)



   RINGKASAN SESI LIVE
  Rep Terdeteksi  : 5
  ROM Rata-rata   : 154.1°  (Min:116° / Maks:205°)
  CSV disimpan di: ./outputs_live/
